# P8-2 · MLOps

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q wandb

## Lý thuyết

In [ ]:
import wandb, math
def train():                                   # thay cho vòng train thật: trả về (bước, loss)
    for step in range(100):
        yield step, 2.0 * math.exp(-step / 30) + 0.1
f1 = 0.83

# mode="offline": chạy không cần tài khoản, log lưu ở thư mục wandb/; bỏ đi để đẩy lên trang W&B
run = wandb.init(project="lab08-extract", mode="offline", config={"lr": 2e-4, "r": 16, "epochs": 3, "data": "v3"})
for step, loss in train():
    wandb.log({"train/loss": loss}, step=step)
wandb.log({"test/f1": f1})
run.finish()
import os
print("đã ghi log vào", os.path.relpath(run.dir))

đã ghi log vào wandb/offline-run-20261106_092346-4w7u3znx/files


In [ ]:
import numpy as np
def gate(scores_new, scores_base, max_drop=0.0):
    d = np.asarray(scores_new, float) - np.asarray(scores_base, float)   # chênh lệch từng mẫu, cùng thứ tự
    mean, se = d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    upper = mean + 1.96 * se                                            # cận trên khoảng tin cậy 95%
    return ("CHẶN" if upper < -max_drop else "QUA"), round(mean, 4), round(upper, 4)

# thử: 300 mẫu, model mới sai thêm 30 mẫu so với baseline
rng = np.random.default_rng(0)
base = (rng.random(300) < 0.8).astype(float)
new = base.copy(); new[:30] = 0
print(gate(new, base))                 # tụt thật: chặn
print(gate(base, base))                # như nhau: qua

('CHẶN', -0.0733, -0.0438)
('QUA', 0.0, 0.0)


## Bài tập 5: Workflow CI có cổng eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
%%writefile eval_gate.py
# eval_gate.py: chạy bộ eval nhỏ, so với baseline đã lưu, thoát mã 1 nếu tụt quá ngưỡng
import json, sys

def main(results_path="eval_results.json", baseline_path="baseline.json", max_drop=0.01):
    cur = json.load(open(results_path))
    base = json.load(open(baseline_path))
    failed = []
    for metric, base_val in base.items():
        drop = base_val - cur.get(metric, 0)
        status = "OK" if drop <= max_drop else "TỤT"
        print(f"{metric:12s} baseline {base_val:.3f} | mới {cur.get(metric, 0):.3f} | {status}")
        if drop > max_drop:
            failed.append(metric)
    sys.exit(1 if failed else 0)

if __name__ == "__main__":
    main()

Writing eval_gate.py


In [ ]:
import json
json.dump({"f1": 0.84, "json_valid": 0.99}, open("baseline.json", "w"))
json.dump({"f1": 0.81, "json_valid": 0.99}, open("eval_results.json", "w"))

In [ ]:
!python eval_gate.py; echo "mã thoát: $?"

f1           baseline 0.840 | mới 0.810 | TỤT
json_valid   baseline 0.990 | mới 0.990 | OK
mã thoát: 1
